In [ ]:
!pip install -q python-dotenv chromadb open-clip-torch transformers accelerate bitsandbytes pandas sentence-transformers

In [ ]:
import os
import warnings
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer, util

import chromadb
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from chromadb.utils.data_loaders import ImageLoader

In [ ]:
warnings.filterwarnings("ignore")
load_dotenv()

from google.colab import drive
drive.mount('/content/drive')

DATASET_FOLDER = "/content/drive/MyDrive/Datasets/menu_image"
EXCEL_PATH = "/content/drive/MyDrive/Datasets/menu_description.xlsx"
DB_IMAGE_PATH = "./data/image_db"
DB_TEXT_PATH = "./data/text_db"
os.makedirs("./data", exist_ok=True)

df = pd.read_excel(EXCEL_PATH)
df['filename'] = df['filename'].astype(str)

In [ ]:
clip_embedder = OpenCLIPEmbeddingFunction()
text_embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
image_loader = ImageLoader()
chroma_client_img = chromadb.PersistentClient(path=DB_IMAGE_PATH)
chroma_client_txt = chromadb.PersistentClient(path=DB_TEXT_PATH)

image_collection = chroma_client_img.get_or_create_collection("image_collection", embedding_function=clip_embedder, data_loader=image_loader)
text_collection = chroma_client_txt.get_or_create_collection("text_collection")

In [ ]:
if image_collection.count() == 0:
    img_ids, img_uris, img_metas = [], [], []
    for i, filename in enumerate(sorted(os.listdir(DATASET_FOLDER))):
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            path = os.path.join(DATASET_FOLDER, filename)
            match = df[df['filename'].str.lower() == filename.lower()]
            description = match['description'].values[0] if not match.empty else "No description"
            ingredients = match['ingredients'].values[0] if not match.empty and 'ingredients' in match.columns else ""

            img_ids.append(str(i))
            img_uris.append(path)
            img_metas.append({"filename": filename, "description": description, "ingredients": ingredients})

    image_collection.add(ids=img_ids, uris=img_uris, metadatas=img_metas)
    print(f"{len(img_ids)} images added to image DB.")

if text_collection.count() == 0:
    text_ids, texts, text_metas = [], [], []
    for i, row in df.iterrows():
        if 'ingredients' in row and not pd.isna(row['ingredients']):
            full_text = f"{row['description']}. Ingredients: {row['ingredients']}"
        else:
            full_text = row['description']

        text_ids.append(str(i))
        texts.append(full_text)
        text_metas.append({
            "filename": row['filename'],
            "description": row['description'],
            "ingredients": row['ingredients'] if 'ingredients' in row else ""
        })

    embeddings = text_embedder.encode(texts).tolist()
    text_collection.add(ids=text_ids, embeddings=embeddings, metadatas=text_metas)
    print(f"{len(texts)} texts added to text DB.")

In [ ]:
def show_image_from_uri(uri):
    img = Image.open(uri)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
def query_image_db(query, top_k=5):
    return image_collection.query(query_texts=[query], n_results=top_k, include=["uris", "metadatas"])

def query_text_db(query, top_k=5):
    query_emb = text_embedder.encode([query]).tolist()
    return text_collection.query(query_embeddings=query_emb, n_results=top_k, include=["metadatas"])

In [ ]:
print("Welcome to Multimodal Restaurants Food Recommender")
query = input("Enter your food craving (e.g., sweet, spicy with tofu): ")

img_results = query_image_db(query, top_k=5)
txt_results = query_text_db(query, top_k=5)

# Extract filenames from both results
img_files = set(meta["filename"] for meta in img_results["metadatas"][0])
txt_files = set(meta["filename"] for meta in txt_results["metadatas"][0])

# Perform late fusion (intersection)
intersect_files = img_files & txt_files
print(f"Found {len(intersect_files)} items in both image and text search.")

for meta in img_results["metadatas"][0]:
    if meta["filename"] in intersect_files:
        path = [u for u, m in zip(img_results["uris"][0], img_results["metadatas"][0]) if m["filename"] == meta["filename"]][0]

        print(f"\n--- {meta['filename']} ---")
        show_image_from_uri(path)

        print(f"Description: {meta['description']}")
        print(f"Ingredients: {meta.get('ingredients', 'Not available')}")

In [ ]:
cosine_scores = []

for meta in img_results["metadatas"][0]:
    filename = meta["filename"]
    if filename in intersect_files:
        full_text = f"{meta['description']}. Ingredients: {meta.get('ingredients', '')}"
        emb_query = text_embedder.encode([query], convert_to_tensor=True)
        emb_item = text_embedder.encode([full_text], convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(emb_query, emb_item).item()
        cosine_scores.append((filename, similarity))

cosine_scores.sort(key=lambda x: x[1], reverse=True)

print("\nRanked Cosine Similarity (Top Results):")
for filename, sim in cosine_scores:
    print(f" {filename}: {sim:.4f}")

In [ ]:
img_files = set(meta["filename"] for meta in img_results["metadatas"][0])
txt_files = set(meta["filename"] for meta in txt_results["metadatas"][0])

intersect_files = img_files & txt_files

match_count = len(intersect_files)

print("\nVisually matching items (CLIP):")
for filename in sorted(img_files):
    print("  -", filename)

print("\nDescription/ingredient matching items (SentenceTransformer):")
for filename in sorted(txt_files):
    print("  -", filename)

print(f"\nLate Fusion Match Count: {match_count} item(s) matched between image and text search results.")
if match_count > 0:
    print("Items that passed late fusion:")
    for f in sorted(intersect_files):
        print("  -", f)
else:
    print("No matching items were found in both modalities.")

